In [1]:
  # ─────────────────────────────────────────────────────────────
  #  CELDA 1: Imports y Configuración Global
  # ─────────────────────────────────────────────────────────────
  import numpy as np
  import cv2
  import matplotlib.pyplot as plt
  import matplotlib.patches as mpatches
  from scipy.signal import correlate
  from dataclasses import dataclass, field
  from typing import Literal
  import os

  # ── Constantes de modulación ──────────────────────────────────
  ASK4_LEVELS   = np.array([0, 85, 170, 255], dtype=np.uint8)  # 4-ASK
  BPSK_CHIPS    = {0: [0, 1], 1: [1, 0]}                       # Manchester
  PILOT_VALUE   = 128                                           # gris medio conocido
  MARKER_WHITE  = 255
  MARKER_BLACK  = 0

  # ── Configuración del módem ───────────────────────────────────
  @dataclass
  class ModemConfig:
      # Grilla de datos (sin contar celdas de marcadores/pilotos)
      M: int = 16          # filas de macropíxeles
      N: int = 16          # columnas de macropíxeles
      cell_size: int = 40  # píxeles por lado de cada celda

      # Modulación
      scheme: Literal["BPSK_Manchester", "4ASK"] = "BPSK_Manchester"

      # Estructura de trama
      marker_cells: int = 3    # tamaño del finder pattern en celdas (3×3)
      pilot_period: int = 4    # insertar piloto cada N celdas de datos

      # Temporización
      symbol_duration_ms: int = 100
      screen_fps: int = 60
      camera_fps: int = 30

      # ── Propiedades derivadas ──────────────────────────────────
      @property
      def frame_width(self) -> int:
          return self.N * self.cell_size

      @property
      def frame_height(self) -> int:
          return self.M * self.cell_size

      @property
      def bits_per_symbol(self) -> int:
          return 1 if self.scheme == "BPSK_Manchester" else 2

      @property
      def chips_per_symbol(self) -> int:
          """Chips que ocupa un símbolo en la grilla (Manchester duplica)."""
          return 2 if self.scheme == "BPSK_Manchester" else 1

      @property
      def total_cells(self) -> int:
          return self.M * self.N

      @property
      def reserved_cells(self) -> int:
          """Celdas ocupadas por los 3 marcadores finder."""
          return 3 * (self.marker_cells ** 2)

      @property
      def available_data_cells(self) -> int:
          return self.total_cells - self.reserved_cells

      @property
      def max_payload_bits(self) -> int:
          """Bits de usuario máximos por trama (descontando pilotos)."""
          pilot_count  = self.available_data_cells // self.pilot_period
          payload_cells = self.available_data_cells - pilot_count
          # En Manchester, cada celda transporta 1 chip = 0.5 bits efectivos
          if self.scheme == "BPSK_Manchester":
              return (payload_cells // 2) * 1   # 2 chips → 1 bit
          return payload_cells * self.bits_per_symbol

      def summary(self):
          print(f"{'─'*46}")
          print(f"  Esquema        : {self.scheme}")
          print(f"  Grilla         : {self.M}×{self.N}  ({self.total_cells} celdas)")
          print(f"  Tamaño celda   : {self.cell_size}×{self.cell_size} px")
          print(f"  Imagen         : {self.frame_width}×{self.frame_height} px")
          print(f"  Celdas libres  : {self.available_data_cells}")
          print(f"  Payload máx.   : {self.max_payload_bits} bits  "
                f"({self.max_payload_bits // 8} bytes)")
          print(f"  Duración símbolo: {self.symbol_duration_ms} ms")
          print(f"{'─'*46}")


  # ── Instancia por defecto ─────────────────────────────────────
  cfg = ModemConfig()
  cfg.summary()

──────────────────────────────────────────────
  Esquema        : BPSK_Manchester
  Grilla         : 16×16  (256 celdas)
  Tamaño celda   : 40×40 px
  Imagen         : 640×640 px
  Celdas libres  : 229
  Payload máx.   : 86 bits  (10 bytes)
  Duración símbolo: 100 ms
──────────────────────────────────────────────
